# 04 — Transfer Learning with MobileNetV2
## Reuse learned visual features

We will use MobileNetV2 pretrained on ImageNet as a feature extractor, attach a small classifier for CIFAR-10, then demonstrate fine-tuning.

**Teaching note:** CIFAR-10 images are small (32×32). For this demonstration we resize a subset to 96×96. Resizing changes the input dimensions but does not create new visual detail. This is a teaching demonstration, not a benchmark.

## 1. Imports and load a small balanced subset

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

tf.keras.utils.set_random_seed(42)
(x_train_all, y_train_all), (x_test_all, y_test_all) = tf.keras.datasets.cifar10.load_data()

# Keep runtime manageable: 500 examples per class for training,
# 100 per class for validation, and 100 per class for testing.
train_idx, val_idx, test_idx = [], [], []
for class_id in range(10):
    idx = np.where(y_train_all.reshape(-1) == class_id)[0]
    train_idx.extend(idx[:500])
    val_idx.extend(idx[500:600])
    test_class_idx = np.where(y_test_all.reshape(-1) == class_id)[0]
    test_idx.extend(test_class_idx[:100])

x_train, y_train = x_train_all[train_idx], y_train_all[train_idx]
x_val, y_val = x_train_all[val_idx], y_train_all[val_idx]
x_test, y_test = x_test_all[test_idx], y_test_all[test_idx]

print("Train:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

## 2. Resize and preprocess for MobileNetV2

MobileNetV2 expects 3-channel images and is commonly used with 224×224 inputs, though its convolutional base supports other suitable spatial sizes. We use 96×96 to reduce workshop runtime.

`mobilenet_v2.preprocess_input` maps pixel values from 0–255 to approximately -1 to 1. Do not divide by 255 before calling it.

In [ ]:
IMAGE_SIZE = (96, 96)

def prepare_images(images):
    images = tf.image.resize(images, IMAGE_SIZE)
    return tf.keras.applications.mobilenet_v2.preprocess_input(images)

x_train = prepare_images(x_train)
x_val = prepare_images(x_val)
x_test = prepare_images(x_test)

print("Resized training tensor:", x_train.shape)
print("Value range:", float(tf.reduce_min(x_train)), float(tf.reduce_max(x_train)))

## 3. Load pretrained MobileNetV2

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False
base_model.summary()

### What do these arguments mean?
- `input_shape`: height, width and channels expected by the model.
- `include_top=False`: remove the original ImageNet classification head.
- `weights="imagenet"`: load weights learned from ImageNet.
- `trainable=False`: freeze the pretrained base; its weights are not updated during this training stage.

MobileNetV2 uses inverted residual blocks and depthwise separable convolutions to reduce computation. We only need the high-level idea here: it is a compact CNN designed for efficient vision.

## 4. Add a new classification head

In [ ]:
inputs = tf.keras.Input(shape=(96, 96, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(10, activation="softmax")(x)

transfer_model = tf.keras.Model(inputs, outputs)
transfer_model.summary()

### Understand the new layers
- `GlobalAveragePooling2D`: averages each feature map spatially, turning H×W×C into C values.
- `Dropout(0.2)`: randomly zeros 20% of activations during training as regularization.
- `Dense(10, softmax)`: produces probabilities for the 10 CIFAR-10 classes.

Compare total parameters with trainable parameters. The frozen base contributes many non-trainable parameters.

## 5. Train the new head

In [ ]:
transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_head = transfer_model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=3,
    batch_size=32
)

## 6. Fine-tuning — unfreeze selected upper layers

In [ ]:
base_model.trainable = True

# Freeze most of the backbone; fine-tune only its final layers.
for layer in base_model.layers[:-20]:
    layer.trainable = False

print("Trainable base layers:", sum(layer.trainable for layer in base_model.layers))
print("Trainable parameters:", sum(int(tf.size(v)) for v in transfer_model.trainable_variables))

Fine-tuning changes pretrained weights, so use a much smaller learning rate. Recompile after changing `trainable` flags so Keras rebuilds the training function with the correct trainable variables.

In [ ]:
transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = transfer_model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=2,
    batch_size=32
)

## 7. Evaluate on held-out test subset

In [ ]:
test_loss, test_accuracy = transfer_model.evaluate(x_test, y_test, verbose=0)
print(f"Transfer-learning test loss: {test_loss:.4f}")
print(f"Transfer-learning test accuracy: {test_accuracy:.4f}")

## Common shape and configuration errors
- Input channel mismatch: MobileNetV2 expects 3 channels; grayscale input needs an explicit compatible strategy.
- Wrong input size: model input shape and actual tensor shape must agree.
- Wrong preprocessing: do not mix `[0,1]` normalization with MobileNetV2's `preprocess_input` expectation.
- Wrong output units: the new Dense layer should match the number of target classes.
- Wrong loss/labels pairing: sparse categorical crossentropy expects integer class IDs; categorical crossentropy expects one-hot labels.
- Forgetting to recompile after changing trainable layers means the intended fine-tuning configuration may not be applied.

## Discussion
1. Why can pretrained features help when the target dataset is small?
2. Why freeze the feature extractor initially?
3. Why use a smaller learning rate for fine-tuning?
4. Why does resizing not add true detail?
5. Compare the parameter counts of the small CNN and MobileNetV2 head/backbone.